# Honest-GAP Prediction IdentityThis notebook demonstrates why ordinary RF-GAP proximities are not prediction-preserving for `HonestForestClassifier`, then checks the Honest-specific `prox_method="honest_gap"` adapter.

In [ ]:
from forestgap import Proximity, embed, get_prototypes, impute, local_feature_importance, outlier_scoresclass _LegacyProximityModel:    def __init__(self, proximity):        self._proximity = proximity    def fit(self, X, y=None, **fit_kwargs):        self._proximity.fit(X, y, **fit_kwargs)        return self    def attach(self, X, y=None):        self._proximity.attach(X, y)        return self    def set_params(self, **params):        prox_params = {}        for key in ("relation", "prox_method", "matrix_type", "non_zero_diagonal", "symm_mode", "max_normalize", "random_state"):            if key in params:                prox_params[key] = params.pop(key)        if params:            self._proximity.estimator.set_params(**params)        if "prox_method" in prox_params and "relation" not in prox_params:            prox_params["relation"] = prox_params.pop("prox_method")        if prox_params:            rel = prox_params.get("relation", self._proximity.relation)            self._proximity._init_sparse_proximity(                prox_method=rel,                matrix_type=prox_params.get("matrix_type", self._proximity.matrix_type),                non_zero_diagonal=prox_params.get("non_zero_diagonal", self._proximity.non_zero_diagonal),                symm_mode=prox_params.get("symm_mode", self._proximity.symm_mode),                max_normalize=prox_params.get("max_normalize", self._proximity.max_normalize),            )            if "random_state" in prox_params:                self._proximity.random_state = prox_params["random_state"]        return self    def get_params(self, deep=True):        params = {}        try:            params.update(self._proximity.estimator.get_params(deep=deep))        except Exception:            pass        params.update({            "relation": self._proximity.relation,            "prox_method": self._proximity.relation,            "matrix_type": self._proximity.matrix_type,            "non_zero_diagonal": self._proximity.non_zero_diagonal,            "symm_mode": self._proximity.symm_mode,            "max_normalize": self._proximity.max_normalize,        })        return params    def phate(self, **kwargs):        return embed(self._proximity, method="phate", **kwargs)    def outlier_scores(self, *args, **kwargs):        return outlier_scores(self._proximity, *args, **kwargs)    def impute(self, *args, **kwargs):        return impute(self._proximity, *args, **kwargs)    def get_prototypes(self, *args, **kwargs):        return get_prototypes(self._proximity, *args, **kwargs)    def local_feature_importance(self, *args, **kwargs):        return local_feature_importance(self._proximity, *args, **kwargs)    def local_variable_importance(self, *args, **kwargs):        return local_feature_importance(self._proximity, *args, **kwargs)    def __getattr__(self, name):        return getattr(self._proximity, name)def compose_proximity(estimator_or_cls):    if isinstance(estimator_or_cls, type):        def _factory(*args, **kwargs):            relation = kwargs.pop("relation", kwargs.pop("prox_method", "rfgap"))            matrix_type = kwargs.pop("matrix_type", "sparse")            non_zero_diagonal = kwargs.pop("non_zero_diagonal", False)            symm_mode = kwargs.pop("symm_mode", None)            max_normalize = kwargs.pop("max_normalize", False)            random_state = kwargs.pop("random_state", None)            estimator = estimator_or_cls(*args, **kwargs)            prox = Proximity(                estimator,                relation=relation,                matrix_type=matrix_type,                non_zero_diagonal=non_zero_diagonal,                symm_mode=symm_mode,                max_normalize=max_normalize,                random_state=random_state,            )            return _LegacyProximityModel(prox)        return _factory    prox = Proximity(estimator_or_cls)    return _LegacyProximityModel(prox)

Vanilla RF-GAP assigns training-sample weights from bootstrap in-bag multiplicities within terminal leaves. Honest forests use one subset to learn split structure and a separate honest subset to estimate leaf predictions. Therefore the fitted leaf posterior is not the empirical distribution of all in-bag samples in the leaf. Honest-GAP instead distributes each fitted leaf/class mass across the honest estimation samples that contributed that class mass.

In [1]:
from pathlib import Pathimport inspectimport sysimport warningsimport numpy as npimport pandas as pdfrom scipy import sparsefrom sklearn.datasets import make_classificationfrom sklearn.model_selection import train_test_splitROOT = Path.cwd()if (ROOT / "forestgap").exists():    sys.path.insert(0, str(ROOT))elif (ROOT.parent / "forestgap").exists():    sys.path.insert(0, str(ROOT.parent))try:    from treeple import HonestForestClassifier    HAS_TREEPLE = True    TREEPLE_ERROR = Noneexcept Exception as exc:    HonestForestClassifier = None    HAS_TREEPLE = False    TREEPLE_ERROR = excif not HAS_TREEPLE:    print(f"Skipping HonestForest demo because treeple is unavailable: {TREEPLE_ERROR}")

In [2]:
def to_dense(mat):    return mat.toarray() if sparse.isspmatrix(mat) else np.asarray(mat)def one_hot(y, classes):    mapping = {cls: i for i, cls in enumerate(np.asarray(classes).tolist())}    y_idx = np.array([mapping[value] for value in y], dtype=np.int64)    return sparse.csc_matrix(        (np.ones(y_idx.shape[0]), (np.arange(y_idx.shape[0]), y_idx)),        shape=(y_idx.shape[0], len(classes)),    )def finite_rows(*arrays):    mask = None    for arr in arrays:        arr = np.asarray(arr, dtype=float)        finite = np.isfinite(arr) if arr.ndim == 1 else np.all(np.isfinite(arr), axis=1)        mask = finite if mask is None else (mask & finite)    return np.asarray(mask, dtype=bool)def diff_summary(name, lhs, rhs):    lhs = np.asarray(lhs, dtype=float)    rhs = np.asarray(rhs, dtype=float)    mask = finite_rows(lhs, rhs)    if not np.any(mask):        return {"comparison": name, "finite_rows": 0, "max_abs_diff": np.nan, "mean_abs_diff": np.nan}    diff = np.abs(lhs[mask] - rhs[mask])    return {        "comparison": name,        "finite_rows": int(mask.sum()),        "max_abs_diff": float(diff.max()),        "mean_abs_diff": float(diff.mean()),    }def honest_model_kwargs(*, prox_method, random_state, kernel_method=True, stratify=True):    sig = inspect.signature(HonestForestClassifier.__init__)    kwargs = {        "n_estimators": 80,        "random_state": random_state,        "prox_method": prox_method,        "matrix_type": "sparse",    }    optional = {        "bootstrap": True,        "oob_score": True,        "honest_fraction": 0.5,        "max_depth": 6,        "min_samples_leaf": 2,        "kernel_method": kernel_method,        "stratify": stratify,    }    for key, value in optional.items():        if key in sig.parameters:            kwargs[key] = value    return kwargs

In [3]:
if HAS_TREEPLE:    X, y = make_classification(        n_samples=260,        n_features=10,        n_informative=7,        n_redundant=1,        n_classes=3,        n_clusters_per_class=1,        class_sep=1.4,        random_state=201,    )    X_train, X_test, y_train, y_test = train_test_split(        X, y, test_size=0.25, stratify=y, random_state=202    )    ProxHonest = compose_proximity(HonestForestClassifier)    print(X_train.shape, X_test.shape, np.bincount(y_train))

(195, 10) (65, 10) [65 67 63]


## Vanilla RF-GAPUsing `prox_method="rfgap"` with an Honest forest now emits a warning. The adapter still computes ordinary RF-GAP, but those weights are not expected to reconstruct Honest forest predictions.

In [4]:
rows = []if HAS_TREEPLE:    with warnings.catch_warnings(record=True) as caught:        warnings.simplefilter("always")        rfgap = ProxHonest(**honest_model_kwargs(            prox_method="rfgap",            random_state=301,            kernel_method=True,            stratify=True,        ))        rfgap.fit(X_train, y_train)    print("Warnings:")    for warning in caught:        print(f"- {warning.message}")    rows.append(diff_summary(        "rfgap prox_predict(X_test) vs predict_proba(X_test)",        rfgap.prox_predict(X_test),        rfgap.predict_proba(X_test),    ))pd.DataFrame(rows)

Warnings:
- prox_method='rfgap' uses ordinary RF-GAP bootstrap semantics, which do not reconstruct HonestForestClassifier predictions. Use prox_method='honest_gap' for HonestForest prediction reconstruction.


,comparison,finite_rows,max_abs_diff,mean_abs_diff
0,rfgap prox_predict(X_test) vs predict_proba(X_...,65,0.097964,0.020933


## Honest-GAP`honest_gap` builds its training-side factor from the honest estimation subset and fitted leaf values. For test data, rows are normalized after extension so `prox_extend(X_test) @ y_one_hot` matches `predict_proba(X_test)`.

In [5]:
if HAS_TREEPLE:    honest_gap = ProxHonest(**honest_model_kwargs(        prox_method="honest_gap",        random_state=302,        kernel_method=True,        stratify=True,    ))    honest_gap.fit(X_train, y_train)    native_test = honest_gap.predict_proba(X_test)    prox_test = honest_gap.prox_predict(X_test)    matrix_test = to_dense(honest_gap.prox_extend(X_test)) @ one_hot(y_train, honest_gap.classes_)    rows.extend([        diff_summary("honest_gap prox_predict(X_test) vs predict_proba(X_test)", prox_test, native_test),        diff_summary("honest_gap prox_extend(X_test) @ y_one_hot vs predict_proba(X_test)", matrix_test, native_test),    ])pd.DataFrame(rows)

,comparison,finite_rows,max_abs_diff,mean_abs_diff
0,rfgap prox_predict(X_test) vs predict_proba(X_...,65,9.796371e-02,2.093274e-02
1,honest_gap prox_predict(X_test) vs predict_pro...,65,4.440892e-16,5.988799e-17
2,honest_gap prox_extend(X_test) @ y_one_hot vs ...,65,3.330669e-16,6.394458e-17


## OOB CaveatFor `kernel_method=True`, treeple's native Honest forest OOB output can be raw/count-like rather than probability-normalized. `honest_gap.prox_predict()` intentionally matches that native OOB scale for training/OOB predictions.

In [6]:
if HAS_TREEPLE:    native_oob = np.asarray(honest_gap.oob_decision_function_, dtype=float)    prox_oob = np.asarray(honest_gap.prox_predict(), dtype=float)    rows.append(diff_summary("honest_gap prox_predict() vs oob_decision_function_", prox_oob, native_oob))    finite = finite_rows(native_oob)    print("Finite OOB rows:", int(finite.sum()))    print("OOB row-sum range:", (float(np.nanmin(native_oob[finite].sum(axis=1))), float(np.nanmax(native_oob[finite].sum(axis=1)))))pd.DataFrame(rows)

Finite OOB rows: 180
OOB row-sum range: (12.76923076923077, 42.66666666666667)


,comparison,finite_rows,max_abs_diff,mean_abs_diff
0,rfgap prox_predict(X_test) vs predict_proba(X_...,65,9.796371e-02,2.093274e-02
1,honest_gap prox_predict(X_test) vs predict_pro...,65,4.440892e-16,5.988799e-17
2,honest_gap prox_extend(X_test) @ y_one_hot vs ...,65,3.330669e-16,6.394458e-17
3,honest_gap prox_predict() vs oob_decision_func...,180,1.752377e-06,2.206434e-07


## Probability-Scaled OOB ModeWith `kernel_method=False`, Honest-GAP uses per-tree leaf probabilities, so OOB rows match the native probability-scaled `oob_decision_function_` on finite rows.

In [7]:
if HAS_TREEPLE and "kernel_method" in inspect.signature(HonestForestClassifier.__init__).parameters:    honest_gap_prob = ProxHonest(**honest_model_kwargs(        prox_method="honest_gap",        random_state=303,        kernel_method=False,        stratify=True,    ))    honest_gap_prob.fit(X_train, y_train)    rows.append(diff_summary(        "honest_gap kernel_method=False prox_predict() vs oob_decision_function_",        honest_gap_prob.prox_predict(),        honest_gap_prob.oob_decision_function_,    ))pd.DataFrame(rows).sort_values("comparison").reset_index(drop=True)

,comparison,finite_rows,max_abs_diff,mean_abs_diff
0,honest_gap kernel_method=False prox_predict() ...,181,4.855087e-08,7.169297e-09
1,honest_gap prox_extend(X_test) @ y_one_hot vs ...,65,3.330669e-16,6.394458e-17
2,honest_gap prox_predict() vs oob_decision_func...,180,1.752377e-06,2.206434e-07
3,honest_gap prox_predict(X_test) vs predict_pro...,65,4.440892e-16,5.988799e-17
4,rfgap prox_predict(X_test) vs predict_proba(X_...,65,9.796371e-02,2.093274e-02


In [8]:
honest_gap_prob.oob_decision_function_

array([[0.03417156, 0.9393921 , 0.02643634],
       [0.74142781, 0.06869778, 0.18987441],
       [0.12804985, 0.78125802, 0.09069212],
       [0.8652696 , 0.1025382 , 0.0321922 ],
       [0.05223051, 0.82908381, 0.11868568],
       [0.75578843, 0.12338103, 0.12083054],
       [0.76887633, 0.16011076, 0.07101291],
       [       nan,        nan,        nan],
       [       nan,        nan,        nan],
       [       nan,        nan,        nan],
       [0.7921421 , 0.15879856, 0.04905934],
       [0.69485267, 0.08336946, 0.22177787],
       [0.79977271, 0.07638694, 0.12384035],
       [0.85194853, 0.09094422, 0.05710725],
       [0.03959747, 0.79351942, 0.16688312],
       [0.0431978 , 0.93179324, 0.02500896],
       [0.26433085, 0.01501502, 0.72065414],
       [0.68446725, 0.24993937, 0.06559338],
       [0.8944162 , 0.06762439, 0.03795941],
       [0.03063439, 0.05110329, 0.91826232],
       [0.82867141, 0.09748412, 0.07384447],
       [0.05933014, 0.8557711 , 0.08489876],
       [0.

In [9]:
honest_gap_prob.prox_predict()

array([[0.03417156, 0.9393921 , 0.02643634],
       [0.74142781, 0.06869778, 0.18987441],
       [0.12804986, 0.78125806, 0.09069213],
       [0.8652696 , 0.1025382 , 0.0321922 ],
       [0.05223051, 0.82908385, 0.11868568],
       [0.75578847, 0.12338104, 0.12083055],
       [0.76887637, 0.16011077, 0.07101292],
       [0.42562876, 0.04757785, 0.46796986],
       [0.05940733, 0.39250501, 0.49545608],
       [0.16467485, 0.61730683, 0.16538675],
       [0.79214214, 0.15879857, 0.04905934],
       [0.69485269, 0.08336946, 0.22177788],
       [0.79977271, 0.07638694, 0.12384035],
       [0.85194853, 0.09094422, 0.05710725],
       [0.03959747, 0.79351945, 0.16688312],
       [0.0431978 , 0.93179325, 0.02500896],
       [0.26433086, 0.01501502, 0.72065416],
       [0.68446727, 0.24993938, 0.06559339],
       [0.89441621, 0.0676244 , 0.03795941],
       [0.03063439, 0.0511033 , 0.91826233],
       [0.82867141, 0.09748412, 0.07384447],
       [0.05933015, 0.85577112, 0.08489876],
       [0.